# Import Libraries

In [1]:
#!pip install fairlearn

In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
#!pip install -q seaborn
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score
import keras
from keras.layers import *
from keras.models import *
from keras.optimizers import Adam


import warnings
from pandas.errors import PerformanceWarning
warnings.simplefilter(action='ignore', category=PerformanceWarning)

# Prepare Dataset

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit

# ── Run settings ──────────────────────────────────────────────────────────────
# Every configuration is run once per random seed (MIDRC_SEED = 0 .. 4); the seed
# controls the CV folds, the network initialisation and the randomised reductions.
SEED    = int(os.environ.get("MIDRC_SEED", "0"))
P_COMP  = int(os.environ.get("MIDRC_PCA", "16"))      # PCA components of the image embedding
LIBRARY = "Fairlearn"

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# ── MIDRC data: real COVID-19 labels and real (self-reported) demographics ─────
DATA_CSV = os.environ.get("MIDRC_DATA_CSV", r"C:\midrc\dataset_final.csv")
FEATURES = os.environ.get("MIDRC_FEATURES", r"C:\midrc\features_xrv_densenet121.npz")
RESULTS_DIR = os.path.join(os.environ.get("MIDRC_RESULTS", os.path.join(os.getcwd(), "Results")),
                           f"seed_{SEED}", LIBRARY)
os.makedirs(RESULTS_DIR, exist_ok=True)

meta = pd.read_csv(DATA_CSV)
feat = np.load(FEATURES, allow_pickle=True)
emb  = pd.DataFrame(feat["embeddings"], index=feat["fname"])     # 1024-d embedding per image
meta = meta[meta["fname"].isin(emb.index)].reset_index(drop=True)
if os.environ.get("MIDRC_SMOKE"):                                 # optional quick test on a subset
    meta = meta.sample(int(os.environ["MIDRC_SMOKE"]), random_state=42).reset_index(drop=True)

# Status: COVID-19 test result (PCR / antigen) provided by MIDRC
meta['Status'] = np.where(meta['covid19_positive'] == 'Yes', 'Covid', 'Non-COVID')

# Gender: self-reported sex
meta['Gender'] = meta['sex']

# Ethnicity: self-reported race / ethnicity grouped into 5 categories
RACE_GROUPS = {'White': 'White', 'Black or African American': 'Black', 'Asian': 'Asian'}
def ethnicity_group(row):
    if row['ethnicity'] == 'Hispanic or Latino':
        return 'Hispanic'
    return RACE_GROUPS.get(row['race'], 'Others')
meta['Ethnicity'] = meta.apply(ethnicity_group, axis=1)

# Age at imaging (years); missing ages are filled with the median age
meta['Age'] = pd.to_numeric(meta['age_at_imaging'], errors='coerce')
meta['Age'] = meta['Age'].fillna(meta['Age'].median() if meta['Age'].notna().any() else 0).round().astype(int)

# Patient identifier: used to keep all images of a patient in the same CV fold
GROUPS = meta['case_id'].values


def grouped_cv(n_splits=5, **_):
    """Stratified K-fold grouped by patient: no patient appears in two folds.
    Replaces the StratifiedKFold of Version 2; the folds change with SEED."""
    return StratifiedGroupKFold(n_splits=n_splits, shuffle=True,
                                random_state=SEED).split(X, y, GROUPS)


def calibration_split(train_index, cal_size=0.25):
    """Split a training fold (by patient) into a fitting part and a calibration part.
    Post-processors are fitted on the calibration part, never on the test fold."""
    gss = GroupShuffleSplit(n_splits=1, test_size=cal_size, random_state=SEED)
    fit_pos, cal_pos = next(gss.split(train_index, groups=GROUPS[train_index]))
    return np.asarray(train_index)[fit_pos], np.asarray(train_index)[cal_pos]


os.chdir(RESULTS_DIR)        # every file written below goes to Results/seed_<SEED>/<LIBRARY>
print(f"{LIBRARY} | seed {SEED} | {len(meta)} images | {meta['case_id'].nunique()} patients | results -> {RESULTS_DIR}")


# ---------------------------------------------------------------------------
# Early stopping without test-fold leakage.
# Every network below stops on val_loss computed on a random hold-out of
# ES_VAL_FRAC of the TRAINING rows of the current fold (seeded by SEED); the
# test fold is never seen during training.  Rows are permuted first because
# dataset_final.csv is sorted by label.
# ---------------------------------------------------------------------------
ES_VAL_FRAC = 0.2

def fit_es(model, X, y, sample_weight=None, validation_data=None, validation_split=None, **kw):
    X = np.asarray(X, dtype='float32')
    y = np.asarray(y)
    idx = np.random.RandomState(SEED).permutation(len(X))
    k = int(round(ES_VAL_FRAC * len(X)))
    va, tr = idx[:k], idx[k:]
    if sample_weight is None:
        return model.fit(X[tr], y[tr], validation_data=(X[va], y[va]), **kw)
    sw = np.asarray(sample_weight, dtype='float32').ravel()
    return model.fit(X[tr], y[tr], sample_weight=sw[tr],
                     validation_data=(X[va], y[va], sw[va]), **kw)


In [ ]:
print("images:", len(meta), "| patients:", meta['case_id'].nunique(), "| seed:", SEED)
print(meta["Status"].value_counts())

In [ ]:
print(pd.crosstab(meta['Ethnicity'], meta['Status'], margins=True))
print(pd.crosstab(meta['Gender'], meta['Status'], margins=True))

In [ ]:
print(meta.shape)
print(meta['Status'].value_counts())

In [ ]:
meta[['fname', 'Status', 'Age', 'Gender', 'Ethnicity', 'case_id']].head(10)

In [ ]:
meta[['fname', 'Status', 'Age', 'Gender', 'Ethnicity', 'case_id']].head(10)

In [ ]:
# Age, Gender and Ethnicity are the real MIDRC values loaded with the data above.
print(meta[['Age']].describe().T)
print(meta['Gender'].value_counts())
print(meta['Ethnicity'].value_counts())

In [ ]:
meta[['fname', 'Status', 'Age', 'Gender', 'Ethnicity', 'case_id']].head(10)

In [ ]:
meta[['fname', 'Status', 'Age', 'Gender', 'Ethnicity', 'case_id']].head(10)

In [ ]:
# Image normalisation is done by the feature extractor (C:\midrc\extract_features.py):
# each PNG is scaled to the [-1024, 1024] range expected by TorchXRayVision,
# centre-cropped and resized to 224 x 224.

# 1st Model: Feature Extraction (chest-X-ray-pretrained DenseNet-121)

In [ ]:
# Feature extractor: DenseNet-121 from TorchXRayVision (weights "densenet121-res224-all"),
# pretrained on ~800,000 chest radiographs from eight public datasets (NIH, CheXpert,
# MIMIC-CXR, PadChest, ...).  None of these is MIDRC, and the network is used frozen,
# so it never sees our labels; this is why no separate image train/test split is
# needed before the feature extraction step.
# The 1024-d penultimate-layer embedding of every image was computed once by
# C:\midrc\extract_features.py and is loaded in the data cell above (`emb`).
print("embedding:", emb.shape)

In [ ]:
# Features: real demographics + image embedding
x = meta[['Age', 'Gender', 'Ethnicity']].copy()
y = meta['Status'].copy()

In [ ]:
status_mapping = {'Non-COVID': 0, 'Covid': 1}
y = y.map(status_mapping)

In [ ]:
# Image embeddings (frozen, CXR-pretrained DenseNet-121), in the same order as `meta`
intermediate_output = emb.loc[meta['fname']].values.astype('float32')
intermediate_output.shape

## Image embeddings (1024-d, from the pretrained DenseNet-121)

In [ ]:
# (V2 extracted features from a CNN trained here; the embeddings are now pre-computed.)

In [ ]:
# (no image arrays are held in memory any more)

## Reducing the Dimensionality of the Image Embeddings

In [ ]:
from sklearn.decomposition import PCA
p_comp = P_COMP
pca = PCA(n_components=p_comp, random_state=SEED)
x[[f'PCA{i}' for i in range(p_comp)]] = pca.fit_transform(intermediate_output)
print(f"PCA: {p_comp} components, {pca.explained_variance_ratio_.sum():.1%} of the embedding variance")
del intermediate_output

# 2nd Model: Predicting COVID-19 Status (image features + Age, Gender and Ethnicity)

In [ ]:
# (unused)

In [ ]:
X = x.reset_index(drop=True)
y = y.reset_index(drop=True)
assert len(X) == len(y) == len(GROUPS)

In [27]:
# Label Encoding
feats = ['Gender','Ethnicity']
le = LabelEncoder()
for f in feats:
    X[f] = le.fit_transform(X[f])

In [28]:
# ── Results store ───────────────────────────────────────────────────────────
# One tidy row per (Algorithm, Sensitive Feature, Status).
# Metrics stored: Accuracy, Accuracy Difference, Demographic Parity Difference,
# Equalized Odds Difference.
RESULT_COLUMNS = [
    'Algorithm', 'Sensitive Feature', 'Status',
    'Accuracy', 'Accuracy Difference',
    'Demographic Parity Difference', 'Equalized Odds Difference',
    'N Groups',
]

metrics_values = pd.DataFrame(columns=RESULT_COLUMNS)

# Original Model

In [ ]:
# Model
inputs_dense = Input(shape=(X.shape[1],))
x = Dense(8, activation='relu', kernel_initializer='glorot_uniform')(inputs_dense)
#x = Dropout(0.5)(x)
x = Dense(6, activation='relu', kernel_initializer='glorot_uniform')(x)
outputs_dense = Dense(4, activation='softmax', kernel_initializer='glorot_uniform')(x)
dense_model = Model(inputs=inputs_dense, outputs=outputs_dense)

# Save Initial Weights
initial_weights = dense_model.get_weights()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# CV and OOF Predictions
y_pred = pd.DataFrame(0.0,columns=[0,1,2,3],index=X.index)

scores = []                   
for train_index, test_index in grouped_cv(n_splits=5):
    X_Train, X_Test = X.iloc[train_index,:], X.iloc[test_index,:]
    y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]
    
    inputs_dense = Input(shape=(X.shape[1],))
    x = Dense(8, activation='relu', kernel_initializer='glorot_uniform')(inputs_dense)
    x = Dense(6, activation='relu', kernel_initializer='glorot_uniform')(x)
    outputs_dense = Dense(4, activation='softmax', kernel_initializer='glorot_uniform')(x)
    dense_model = Model(inputs=inputs_dense, outputs=outputs_dense)
    dense_model.compile(optimizer=Adam(learning_rate=0.01), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    # Reinitialize the weights
    dense_model.set_weights(initial_weights)
    
    # Define early stopping
    es = EarlyStopping(monitor='val_loss', mode='min',patience=3)

    fit_es(dense_model, X_Train.values.astype('float32'), y_Train.values.astype('float32'),
                    epochs=25, batch_size=64, verbose=0,
                    callbacks=[es])  
    preds = dense_model(X_Test.values.astype('float32')).numpy().squeeze()
    y_pred.iloc[test_index,:] = preds
    
    scores.append(accuracy_score(y_Test.values.astype('float32'),np.argmax(preds,axis=1)))
    print(scores[-1])

print("Mean:",np.mean(scores),"\nSTD: ", np.std(scores),'\n')


# 0.7342332841172748
# 0.7449905083315757
# 0.7333895802573297
# 0.7308584686774942
# 0.7373971735920691
# Mean: 0.7361738029951488 
# STD:  0.004878549318893849 

In [ ]:
# Save the out-of-fold predictions of the Original Model and the evaluation frame
# (labels, sensitive attributes, patient id) for the cross-seed analysis notebook.
y_pred.to_csv('y_pred_Original.csv', index=False)
pd.DataFrame({'Status': y.values, 'Age': X['Age'].values, 'Gender': X['Gender'].values,
              'Ethnicity': X['Ethnicity'].values, 'case_id': GROUPS}).to_csv('eval_frame.csv', index=False)

# Metrics

In [33]:
def multiclass_selection_rate(y_true, y_pred, desired_class):
    """Selection rate for one Status class in the multiclass problem.

    The selection rate is the fraction of predicted labels equal to `desired_class`
    (i.e. P[Y_hat = class] within the group). This is the quantity compared across
    groups by the demographic parity difference.

    Args:
        y_true : array-like  -- true labels (unused, kept for the MetricFrame signature)
        y_pred : array-like  -- predicted *labels* (not probabilities)
        desired_class : any  -- the Status class of interest
    Returns:
        float -- selection rate for the desired class
    """
    y_pred = np.asarray(y_pred)
    total = y_pred.shape[0]
    return np.nan if total == 0 else (y_pred == desired_class).sum() / total

In [34]:
from fairlearn.metrics import MetricFrame
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd

# ── Status classes (must match the mapping used for y) ──────────────────────
STATUS_MAPPING = {'Non-COVID': 0, 'Covid': 1}
CLASS_NAMES = {v: k for k, v in STATUS_MAPPING.items()}
N_CLASSES = len(STATUS_MAPPING)

# ── Sensitive-feature configurations to report on ───────────────────────────
SENSITIVE_SPECS = {
    'Gender': ['Gender'],
    'Ethnicity': ['Ethnicity'],
    'Gender + Ethnicity': ['Gender', 'Ethnicity'],
}

# Readable labels for the label-encoded columns
GENDER_LABELS = {0: 'Female', 1: 'Male'}
ETHNICITY_LABELS = {0: 'Asian', 1: 'Black', 2: 'Hispanic', 3: 'Others', 4: 'White'}


def to_labels(y_score, n_classes=N_CLASSES):
    """Convert a score/probability matrix to hard Status labels.

    The networks emit 4 output columns (a leftover from the age-group model), so
    only the first `n_classes` columns - the ones that correspond to actual Status
    values - are considered. A 1-D input is assumed to already be labels.
    """
    arr = np.asarray(y_score.values if hasattr(y_score, 'values') else y_score)
    if arr.ndim == 1:
        return arr.astype(int)
    return np.argmax(arr[:, :n_classes], axis=1).astype(int)


def build_sensitive_feature(X, cols):
    """Return a single Series of (possibly intersectional) group labels."""
    if len(cols) == 1:
        col = cols[0]
        s = pd.Series(np.asarray(X[col]), name=col)
        if col == 'Gender':
            s = s.map(lambda v: GENDER_LABELS.get(v, v))
        elif col == 'Ethnicity':
            s = s.map(lambda v: ETHNICITY_LABELS.get(v, v))
        return s.astype(str)
    parts = [build_sensitive_feature(X, [c]) for c in cols]
    combined = parts[0].str.cat(parts[1:], sep=' | ')
    combined.name = ' + '.join(cols)
    return combined


# ── Per-group rates for a one-vs-rest (binary) view of a Status class ───────
def _selection_rate(y_true, y_pred):
    y_pred = np.asarray(y_pred)
    return np.nan if len(y_pred) == 0 else float((y_pred == 1).mean())


def _tpr(y_true, y_pred):
    """True positive rate; NaN when the group has no positive cases."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    pos = (y_true == 1).sum()
    return np.nan if pos == 0 else float(((y_pred == 1) & (y_true == 1)).sum() / pos)


def _fpr(y_true, y_pred):
    """False positive rate; NaN when the group has no negative cases."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    neg = (y_true == 0).sum()
    return np.nan if neg == 0 else float(((y_pred == 1) & (y_true == 0)).sum() / neg)


BINARY_METRICS = {
    'accuracy': lambda yt, yp: accuracy_score(np.asarray(yt), np.asarray(yp)),
    'selection_rate': _selection_rate,
    'tpr': _tpr,
    'fpr': _fpr,
    'count': lambda yt, yp: len(yt),
}


def _spread(values):
    """max - min across groups, ignoring groups where the rate is undefined."""
    v = pd.to_numeric(pd.Series(list(values)), errors='coerce').dropna()
    return np.nan if len(v) < 2 else float(v.max() - v.min())


# ── Metrics used by the by-group tables (now named after Status) ────────────
metrics = {
    'accuracy': lambda y_true, y_pred: accuracy_score(y_true, to_labels(y_pred)),
    'selection_rate_Non-COVID': lambda y_true, y_pred: multiclass_selection_rate(y_true, to_labels(y_pred), STATUS_MAPPING['Non-COVID']),
    'selection_rate_Covid':     lambda y_true, y_pred: multiclass_selection_rate(y_true, to_labels(y_pred), STATUS_MAPPING['Covid']),
    'count': lambda y_true, y_pred: len(y_true),
}


def evaluate_fairness(y_true, y_score, X, algorithm, store=True, verbose=True, decimals=3):
    """Accuracy, demographic parity difference and equalized odds difference,
    computed for every Status value and every sensitive-feature configuration.

    For each Status class the problem is binarised one-vs-rest, then:
      * Accuracy                      -- accuracy of that one-vs-rest decision
                                         (the 'Overall (all classes)' row is the
                                         plain multiclass accuracy)
      * Accuracy Difference           -- max - min accuracy across groups
      * Demographic Parity Difference -- max - min selection rate across groups
      * Equalized Odds Difference     -- max(TPR spread, FPR spread) across groups

    Groups where a rate is undefined (no positives / no negatives) are skipped
    rather than raising, so small intersectional groups can't break the run.
    """
    global metrics_values

    y_labels = to_labels(y_score)
    y_true_arr = np.asarray(y_true.values if hasattr(y_true, 'values') else y_true).astype(int)

    rows = []
    for sf_name, cols in SENSITIVE_SPECS.items():
        sf = build_sensitive_feature(X, cols)

        # Multiclass accuracy, overall and by group
        mf_all = MetricFrame(
            metrics={'accuracy': lambda yt, yp: accuracy_score(np.asarray(yt), np.asarray(yp))},
            y_true=y_true_arr, y_pred=y_labels, sensitive_features=sf,
        )
        rows.append({
            'Algorithm': algorithm,
            'Sensitive Feature': sf_name,
            'Status': 'Overall (all classes)',
            'Accuracy': accuracy_score(y_true_arr, y_labels),
            'Accuracy Difference': _spread(mf_all.by_group['accuracy']),
            'Demographic Parity Difference': np.nan,
            'Equalized Odds Difference': np.nan,
            'N Groups': int(mf_all.by_group.shape[0]),
        })

        # One row per Status value
        for cls_val in sorted(CLASS_NAMES):
            yt = (y_true_arr == cls_val).astype(int)
            yp = (y_labels == cls_val).astype(int)
            mf = MetricFrame(metrics=BINARY_METRICS, y_true=yt, y_pred=yp, sensitive_features=sf)
            rows.append({
                'Algorithm': algorithm,
                'Sensitive Feature': sf_name,
                'Status': CLASS_NAMES[cls_val],
                'Accuracy': accuracy_score(yt, yp),
                'Accuracy Difference': _spread(mf.by_group['accuracy']),
                'Demographic Parity Difference': _spread(mf.by_group['selection_rate']),
                'Equalized Odds Difference': np.nanmax([
                    _spread(mf.by_group['tpr']),
                    _spread(mf.by_group['fpr']),
                ]),
                'N Groups': int(mf.by_group.shape[0]),
            })

    results = pd.DataFrame(rows, columns=RESULT_COLUMNS)

    if store:
        if len(metrics_values):
            base = metrics_values[metrics_values['Algorithm'] != algorithm]
            metrics_values = (
                pd.concat([base, results], ignore_index=True)
                if len(base) else results.copy()
            )
        else:
            metrics_values = results.copy()

    if verbose:
        print(f'=== {algorithm} ===')
        print(results.round(decimals).to_string(index=False))

    return results


def group_breakdown(y_true, y_score, X, sensitive='Gender', decimals=3):
    """Per-group accuracy / selection rate / TPR / FPR for each Status value.

    These are the raw numbers the differences above are computed from - useful for
    seeing *which* group drives a large disparity.
    """
    y_labels = to_labels(y_score)
    y_true_arr = np.asarray(y_true.values if hasattr(y_true, 'values') else y_true).astype(int)
    sf = build_sensitive_feature(X, SENSITIVE_SPECS[sensitive])

    tables = {}
    for cls_val in sorted(CLASS_NAMES):
        yt = (y_true_arr == cls_val).astype(int)
        yp = (y_labels == cls_val).astype(int)
        mf = MetricFrame(metrics=BINARY_METRICS, y_true=yt, y_pred=yp, sensitive_features=sf)
        tables[CLASS_NAMES[cls_val]] = mf.by_group

    return pd.concat(tables, names=['Status']).round(decimals)


def comparison_table(metric='Equalized Odds Difference', decimals=3):
    """Wide Algorithm-by-(Sensitive Feature, Status) table for one metric."""
    if not len(metrics_values):
        return metrics_values
    return (metrics_values
            .pivot_table(index=['Sensitive Feature', 'Status'],
                         columns='Algorithm', values=metric, sort=False)
            .round(decimals))

### Table for Gender

#### Gender: {0: Female , 1: Male}
#### Ethnicity: {0: Asian , 1: Black , 2: Hispanic , 3: Others , 4: White}
#### Status: {0: Non-COVID , 1: Covid}

Each table below shows, per group: overall accuracy and the selection
rate for each **Status** value (the share of the group predicted into that class).

In [37]:
# Create a MetricFrame object
mf = MetricFrame(metrics=metrics, y_true=y, y_pred=y_pred, sensitive_features=X[['Gender']])

# Compute the metrics for each group defined by the sensitive features
results = {}
for metric_name in metrics.keys():
        results[metric_name] = mf.by_group[metric_name]

# Convert the results to a DataFrame
results_df = pd.DataFrame(results)

# Display the results
results_df


Out[28]: 
        accuracy  selection_rate_Non-COVID  selection_rate_Covid   count
Gender                                                                  
0       0.620139                  0.585764              0.414236  2880.0
1       0.610452                  0.573902              0.426098  3119.0


### Table for Ethnicity

In [39]:
# Create a MetricFrame object
mf = MetricFrame(metrics=metrics, y_true=y, y_pred=y_pred, sensitive_features=X[['Ethnicity']])

# Compute the metrics for each group defined by the sensitive features
results = {}
for metric_name in metrics.keys():
        results[metric_name] = mf.by_group[metric_name]

# Convert the results to a DataFrame
results_df = pd.DataFrame(results)

# Display the results
results_df

Out[29]: 
           accuracy  selection_rate_Non-COVID  selection_rate_Covid   count
Ethnicity                                                                  
0          0.549398                  0.207229              0.792771   415.0
1          0.640730                  0.196926              0.803074  2082.0
2          0.573370                  0.355978              0.644022   368.0
3          0.533784                  0.729730              0.270270   148.0
4          0.615539                  0.918285              0.081715  2986.0


### Table for Age

In [41]:
# Create a MetricFrame object
mf = MetricFrame(metrics=metrics, y_true=y, y_pred=y_pred, sensitive_features=X[['Age']])

# Compute the metrics for each group defined by the sensitive features
results = {}
for metric_name in metrics.keys():
        results[metric_name] = mf.by_group[metric_name]

# Convert the results to a DataFrame
results_df = pd.DataFrame(results)

# Display the results
results_df

Out[30]: 
     accuracy  selection_rate_Non-COVID  selection_rate_Covid  count
Age                                                                 
0    0.941176                  1.000000              0.000000   17.0
1    1.000000                  1.000000              0.000000    3.0
2    0.500000                  0.500000              0.500000    2.0
3    0.800000                  0.800000              0.200000    5.0
4    1.000000                  0.500000              0.500000    2.0
..        ...                       ...                   ...    ...
85   0.594203                  0.681159              0.318841   69.0
86   0.700000                  0.900000              0.100000   40.0
87   0.652174                  0.652174              0.347826   46.0
88   0.733333                  0.822222              0.177778   45.0
89   0.593750                  0.875000              0.125000   32.0

[89 rows x 4 columns]


### Table for Gender and Ethnicity

In [43]:
# Create a MetricFrame object
mf = MetricFrame(metrics=metrics, y_true=y, y_pred=y_pred, sensitive_features=X[['Gender','Ethnicity']])

# Compute the metrics for each group defined by the sensitive features
results = {}
for metric_name in metrics.keys():
        results[metric_name] = mf.by_group[metric_name]

# Convert the results to a DataFrame
results_df = pd.DataFrame(results)

# Display the results
results_df

Out[31]: 
                  accuracy  ...   count
Gender Ethnicity            ...        
0      0          0.537234  ...   188.0
       1          0.647348  ...  1018.0
       2          0.534591  ...   159.0
       3          0.491803  ...    61.0
       4          0.626547  ...  1454.0
1      0          0.559471  ...   227.0
       1          0.634398  ...  1064.0
       2          0.602871  ...   209.0
       3          0.563218  ...    87.0
       4          0.605091  ...  1532.0

[10 rows x 4 columns]


### Accuracy, Demographic Parity & Equalized Odds by Status

For every Status value (Normal / Covid / Pneumonia) and every sensitive-feature
configuration (Gender, Ethnicity, Gender + Ethnicity).

In [45]:
import warnings
from pandas.errors import PerformanceWarning

warnings.simplefilter(action='ignore', category=PerformanceWarning)

In [46]:
warnings.resetwarnings()
warnings.simplefilter(action='ignore', category=PerformanceWarning)

In [47]:
results_original = evaluate_fairness(y, y_pred, X, algorithm='Original')
results_original

=== Original ===
Algorithm  Sensitive Feature                Status  Accuracy  Accuracy Difference  Demographic Parity Difference  Equalized Odds Difference  N Groups
 Original             Gender Overall (all classes)     0.615                0.010                            NaN                        NaN         2
 Original             Gender             Non-COVID     0.615                0.010                          0.012                      0.022         2
 Original             Gender                 Covid     0.615                0.010                          0.012                      0.022         2
 Original          Ethnicity Overall (all classes)     0.615                0.107                            NaN                        NaN         5
 Original          Ethnicity             Non-COVID     0.615                0.107                          0.721                      0.760         5
 Original          Ethnicity                 Covid     0.615                0.107  

### Per-group breakdown (Original)

The per-group rates behind the differences above: accuracy, selection rate, TPR
and FPR for each Status value.

In [49]:
# NOTE: superseded by evaluate_fairness() / group_breakdown() above, which handle
# groups that contain only one class instead of raising on confusion_matrix(...).ravel().
# Kept for reference only.
from sklearn.metrics import confusion_matrix


def equalized_odds_difference_legacy(y_test, y_pred, target_class, sensitive_feature):
    tpr, fpr = [], []
    df = pd.concat([y_test, pd.Series(y_pred), sensitive_feature], axis=1).reset_index(drop=True)
    df = df.rename({'Status': 'y_test', 0: 'y_pred'}, axis=1)

    for cls in sensitive_feature.value_counts().keys().values:
        y_test_binary = df.loc[df[sensitive_feature.columns[0]] == cls, 'y_test']
        y_pred_binary = df.loc[df[sensitive_feature.columns[0]] == cls, 'y_pred']
        y_test_binary = (y_test_binary.values == target_class) * 1
        y_pred_binary = (y_pred_binary.values == target_class) * 1
        tn, fp, fn, tp = confusion_matrix(y_test_binary, y_pred_binary).ravel()
        tpr.append(tp / (tp + fn))
        fpr.append(fp / (fp + tn))

    return max(max(tpr) - min(tpr), max(fpr) - min(fpr))

In [50]:
for sf_name in SENSITIVE_SPECS:
    print(f'--- Original | Sensitive Feature: {sf_name} ---')
    display(group_breakdown(y, y_pred, X, sensitive=sf_name))

--- Original | Sensitive Feature: Gender ---
                  accuracy  selection_rate    tpr    fpr   count
Status    Gender                                                
Non-COVID Female      0.62           0.586  0.706  0.466  2880.0
          Male        0.61           0.574  0.684  0.463  3119.0
Covid     Female      0.62           0.414  0.534  0.294  2880.0
          Male        0.61           0.426  0.537  0.316  3119.0
--- Original | Sensitive Feature: Ethnicity ---
                     accuracy  selection_rate    tpr    fpr   count
Status    Ethnicity                                                
Non-COVID Asian         0.549           0.207  0.271  0.134   415.0
          Black         0.641           0.197  0.276  0.152  2082.0
          Hispanic      0.573           0.356  0.386  0.343   368.0
          Others        0.534           0.730  0.767  0.693   148.0
          White         0.616           0.918  0.934  0.894  2986.0
Covid     Asian         0.549           0

## Shared model architecture

Each mitigation section below was defining its own network: `9 -> 6 -> 4` for
Correlation Remover, and `11 -> 8 -> 4 -> 2` for the reductions, Threshold
Optimizer and the sequential blocks. The **Original Model** is `11 -> 8 -> 6 -> 4`.
The cell below extracts that architecture into one factory so every section
trains the same network and the accuracies stay comparable.


In [ ]:
import os
# ===========================================================================
# Shared model architecture
#
# Taken from the "Original Model" section above:
#
#     Input(n_features) -> Dense(8, relu) -> Dense(6, relu) -> Dense(n_out, softmax)
#     Adam(learning_rate=0.01) | 25 epochs | batch 64 | glorot_uniform init
#     EarlyStopping(monitor='val_loss', mode='min', patience=3)
#     the same starting weights are reused for every fold
#
# Every mitigation section below builds its network through this factory, so a
# change in accuracy reflects the mitigation algorithm rather than a different
# network.
#
# Two things still legitimately vary and are read from the data, never hard-coded:
#   input width  - 11 raw features, 9 after CorrelationRemover drops the two
#                  sensitive columns
#   output width - n_outputs=4 is the multiclass head used by the Original Model;
#                  n_outputs=2 is the one-vs-rest head the reductions and
#                  ThresholdOptimizer need (still softmax +
#                  sparse_categorical_crossentropy, as elsewhere in this notebook)
# ===========================================================================
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

ORIG_H1, ORIG_H2 = 8, 6          # hidden widths of the Original Model
ORIG_LR          = 0.01
ORIG_EPOCHS      = 25
ORIG_BATCH_SIZE  = 64
ORIG_PATIENCE    = 3

GS_GRID_SIZE = 40                 # Grid Search grid size (Fairlearn)
if os.environ.get("MIDRC_FAST"):  # quick test runs only
    ORIG_EPOCHS, GS_GRID_SIZE = 2, 3

SENSITIVE_COLS = ['Gender', 'Ethnicity']
N_FEATURES_RAW = X.shape[1]                              # 11
N_FEATURES_CR  = N_FEATURES_RAW - len(SENSITIVE_COLS)    # 9 after CorrelationRemover

_INITIAL_WEIGHTS = {}            # one fixed init per (input_dim, n_outputs)


def build_original_arch(input_dim=None, n_outputs=4, weights=None):
    """Compiled copy of the Original Model architecture, set to the shared
    initial weights for its (input_dim, n_outputs) head unless `weights` is given."""
    input_dim = N_FEATURES_RAW if input_dim is None else int(input_dim)

    inputs = Input(shape=(input_dim,))
    h   = Dense(ORIG_H1, activation='relu', kernel_initializer='glorot_uniform')(inputs)
    h   = Dense(ORIG_H2, activation='relu', kernel_initializer='glorot_uniform')(h)
    out = Dense(n_outputs, activation='softmax', kernel_initializer='glorot_uniform')(h)

    model = Model(inputs=inputs, outputs=out)
    model.compile(optimizer=Adam(learning_rate=ORIG_LR),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    key = (input_dim, n_outputs)
    _INITIAL_WEIGHTS.setdefault(key, model.get_weights())
    model.set_weights(_INITIAL_WEIGHTS[key] if weights is None else weights)
    return model


def original_initial_weights(input_dim=None, n_outputs=4):
    """Shared starting weights for a given head, building them on first use."""
    input_dim = N_FEATURES_RAW if input_dim is None else int(input_dim)
    build_original_arch(input_dim, n_outputs)
    return _INITIAL_WEIGHTS[(input_dim, n_outputs)]


def original_early_stopping(patience=None, restore_best_weights=False):
    """The callback used by the Original Model."""
    return EarlyStopping(monitor='val_loss', mode='min',
                         patience=ORIG_PATIENCE if patience is None else patience,
                         restore_best_weights=restore_best_weights)


print(f"{N_FEATURES_RAW} raw features, {N_FEATURES_CR} after CorrelationRemover.")
print("Architecture used by every section below:")
build_original_arch().summary()


# Mitigation Algorithms

# Preprocessing Algorithms:

In [55]:
# Architecture: the Original Model (n_features -> 8 -> 6 -> 4 softmax).
# CorrelationRemover drops the two sensitive columns, so the input is
# N_FEATURES_CR (9) wide here; the hidden layers are unchanged.
initial_weights = original_initial_weights(N_FEATURES_CR, n_outputs=4)


In [ ]:
from fairlearn.preprocessing import CorrelationRemover

# CV and OOF Predictions
y_pred_1 = pd.DataFrame(0.0,columns=[0,1,2,3],index=X.index)

scores = []                   
for train_index, test_index in grouped_cv(n_splits=5):
    X_Train, X_Test = X.iloc[train_index,:], X.iloc[test_index,:]
    y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]
    
    corr_remover = CorrelationRemover(sensitive_feature_ids=['Gender', 'Ethnicity'])
    corr_remover.fit(X_Train)
    X_Train = corr_remover.transform(X_Train)
    X_Test = corr_remover.transform(X_Test)
    
    # Original Model architecture, compiled and reset to the shared init weights
    dense_model = build_original_arch(N_FEATURES_CR, n_outputs=4)

    # Define early stopping
    es = original_early_stopping()

    fit_es(dense_model, X_Train.astype('float32'), y_Train.values.astype('float32'),
                    epochs=ORIG_EPOCHS, batch_size=ORIG_BATCH_SIZE, verbose=0,
                    callbacks=[es])
    preds = dense_model(X_Test.astype('float32')).numpy().squeeze()
    y_pred_1.iloc[test_index,:] = preds
    
    scores.append(accuracy_score(y_Test.values.astype('float32'),np.argmax(preds,axis=1)))
    print(scores[-1])

print("Mean:",np.mean(scores),"\nSTD: ", np.std(scores),'\n')

In [57]:
y_pred_1.to_csv('y_pred_Corr.csv',index=False)

#### Accuracy, Demographic Parity and Equalized Odds by Status

In [59]:
results_corr = evaluate_fairness(y, y_pred_1, X, algorithm='Corr')
results_corr

=== Corr ===
Algorithm  Sensitive Feature                Status  Accuracy  Accuracy Difference  Demographic Parity Difference  Equalized Odds Difference  N Groups
     Corr             Gender Overall (all classes)     0.542                0.046                            NaN                        NaN         2
     Corr             Gender             Non-COVID     0.542                0.046                          0.006                      0.051         2
     Corr             Gender                 Covid     0.542                0.046                          0.006                      0.051         2
     Corr          Ethnicity Overall (all classes)     0.542                0.081                            NaN                        NaN         5
     Corr          Ethnicity             Non-COVID     0.542                0.081                          0.177                      0.215         5
     Corr          Ethnicity                 Covid     0.542                0.081      

In [60]:
for sf_name in SENSITIVE_SPECS:
    print(f'--- Correlation Remover | Sensitive Feature: {sf_name} ---')
    display(group_breakdown(y, y_pred_1, X, sensitive=sf_name))

--- Correlation Remover | Sensitive Feature: Gender ---
                  accuracy  selection_rate    tpr    fpr   count
Status    Gender                                                
Non-COVID Female     0.566           0.427  0.493  0.361  2880.0
          Male       0.520           0.433  0.453  0.412  3119.0
Covid     Female     0.566           0.573  0.639  0.507  2880.0
          Male       0.520           0.567  0.588  0.547  3119.0
--- Correlation Remover | Sensitive Feature: Ethnicity ---
                     accuracy  selection_rate    tpr    fpr   count
Status    Ethnicity                                                
Non-COVID Asian         0.537           0.533  0.566  0.495   415.0
          Black         0.579           0.379  0.443  0.343  2082.0
          Hispanic      0.552           0.356  0.351  0.358   368.0
          Others        0.595           0.453  0.548  0.360   148.0
          White         0.513           0.458  0.477  0.429  2986.0
Covid     Asian    

# In-processing Algorithms:

In [62]:
# Architecture: the Original Model (n_features -> 8 -> 6), with the 2-unit
# softmax one-vs-rest head this section needs.
initial_weights = original_initial_weights(N_FEATURES_RAW, n_outputs=2)


#### Exponentiated Gradient

In [ ]:
from fairlearn.reductions import ExponentiatedGradient, DemographicParity, EqualizedOdds
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import clone_model

class DeepLearningEstimator:
    def __init__(self, model, initial_weights, target_class=None):
        self.model = model
        self.initial_weights = initial_weights
        self.target_class = target_class
    def get_params(self, deep=True):
        return {
            "model": self.model,
            "initial_weights": self.initial_weights,
            "target_class": self.target_class
        }
    def set_params(self, **params):
        for k, v in params.items():
            setattr(self, k, v)
        return self
    def __sklearn_clone__(self):
        new_model = clone_model(self.model)
        new_model.compile(
            optimizer=Adam(learning_rate=ORIG_LR),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        new_model.set_weights(self.initial_weights)
        return DeepLearningEstimator(new_model, self.initial_weights, self.target_class)
    def fit(self, X, y, sample_weight=None):
        y_val_binary = (y_Test == self.target_class).values.astype('float32')
        es = original_early_stopping()
        fit_es(self.model, X.astype('float32'), y.astype('float32'),
            sample_weight=sample_weight,
            epochs=ORIG_EPOCHS, batch_size=ORIG_BATCH_SIZE, verbose=0,
            callbacks=[es]
        )
    def predict(self, X):
        return self.model(X.astype('float32')).numpy().squeeze()[:, 1]

# CV and OOF Predictions
y_pred_2 = pd.DataFrame(0.0, columns=[0,1,2,3], index=X.index)
scores = []
for train_index, test_index in grouped_cv(n_splits=2):
    X_Train, X_Test = X.iloc[train_index,:], X.iloc[test_index,:]
    y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]

    # ExponentiatedGradient (one-vs-all)
    expgrad_estimators = []
    classes = sorted(CLASS_NAMES)   # the Status classes present in the data
    for cls in classes:
        # Convert multiclass to binary (one-vs-all)
        y_Train_binary = (y_Train == cls).astype(int)

        # ── Fresh model + estimator per class, with target_class=cls ──
        # Original Model architecture with the one-vs-rest head
        dense_model = build_original_arch(N_FEATURES_RAW, n_outputs=2)

        dl_estimator = DeepLearningEstimator(dense_model, initial_weights, target_class=cls)  # ← cls here

        expgrad = ExponentiatedGradient(dl_estimator, constraints=EqualizedOdds(), max_iter=10)
        expgrad.fit(X_Train.values.astype('float32'),
                    y_Train_binary.values.astype('float32'),
                    sensitive_features=X_Train[['Gender','Ethnicity']])
        expgrad_estimators.append(expgrad)

    # Generate OOF predictions for each class
    for i, expgrad in enumerate(expgrad_estimators):
        y_pred_2.iloc[test_index, i] = expgrad.predict(X_Test.values)

    scores.append(accuracy_score(
        y_Test.values.astype('float32'),
        np.argmax(y_pred_2.loc[test_index,:].values, axis=1).astype('float32')
    ))
    print(scores[-1])

print("Mean:", np.mean(scores), "\nSTD: ", np.std(scores), '\n')

In [65]:
y_pred_2.to_csv('y_pred_expGrad.csv',index=False)

In [66]:
results_expon = evaluate_fairness(y, y_pred_2, X, algorithm='Expon')
results_expon

=== Expon ===
Algorithm  Sensitive Feature                Status  Accuracy  Accuracy Difference  Demographic Parity Difference  Equalized Odds Difference  N Groups
    Expon             Gender Overall (all classes)     0.512                0.014                            NaN                        NaN         2
    Expon             Gender             Non-COVID     0.512                0.014                          0.004                      0.019         2
    Expon             Gender                 Covid     0.512                0.014                          0.004                      0.019         2
    Expon          Ethnicity Overall (all classes)     0.512                0.117                            NaN                        NaN         5
    Expon          Ethnicity             Non-COVID     0.512                0.117                          0.055                      0.067         5
    Expon          Ethnicity                 Covid     0.512                0.117     

In [67]:
for sf_name in SENSITIVE_SPECS:
    print(f'--- Exponentiated Gradient | Sensitive Feature: {sf_name} ---')
    display(group_breakdown(y, y_pred_2, X, sensitive=sf_name))

--- Exponentiated Gradient | Sensitive Feature: Gender ---
                  accuracy  selection_rate    tpr    fpr   count
Status    Gender                                                
Non-COVID Female     0.519           0.711  0.730  0.692  2880.0
          Male       0.505           0.707  0.711  0.702  3119.0
Covid     Female     0.519           0.289  0.308  0.270  2880.0
          Male       0.505           0.293  0.298  0.289  3119.0
--- Exponentiated Gradient | Sensitive Feature: Ethnicity ---
                     accuracy  selection_rate    tpr    fpr   count
Status    Ethnicity                                                
Non-COVID Asian         0.552           0.740  0.774  0.701   415.0
          Black         0.442           0.707  0.707  0.708  2082.0
          Hispanic      0.462           0.685  0.737  0.661   368.0
          Others        0.541           0.723  0.767  0.680   148.0
          White         0.559           0.708  0.717  0.693  2986.0
Covid     Asi

#### Grid Search

In [ ]:
from fairlearn.reductions import DemographicParity, GridSearch, EqualizedOdds
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import clone_model

class DeepLearningEstimator:
    def __init__(self, model, initial_weights, target_class=None):
        self.model = model
        self.initial_weights = initial_weights
        self.target_class = target_class

    def get_params(self, deep=True):
        return {
            "model": self.model,
            "initial_weights": self.initial_weights,
            "target_class": self.target_class
        }

    def set_params(self, **params):
        for k, v in params.items():
            setattr(self, k, v)
        return self

    def __sklearn_clone__(self):
        return self.__deepcopy__({})

    def __deepcopy__(self, memo):
        # Called by GridSearch's copy.deepcopy — rebuilds model safely
        new_model = clone_model(self.model)
        new_model.compile(
            optimizer=Adam(learning_rate=ORIG_LR),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        new_model.set_weights(self.initial_weights)
        return DeepLearningEstimator(new_model, self.initial_weights, self.target_class)

    def fit(self, X, y, sample_weight=None):
        y_val_binary = (y_Test == self.target_class).values.astype('float32')
        es = original_early_stopping()
        fit_es(self.model, X.astype('float32'), y.astype('float32'),
            sample_weight=sample_weight,
            epochs=ORIG_EPOCHS, batch_size=ORIG_BATCH_SIZE, verbose=0,
            callbacks=[es]
        )

    def predict(self, X):
        return self.model(X.astype('float32')).numpy().squeeze()[:, 1]


# CV and OOF Predictions
y_pred_3 = pd.DataFrame(0.0, columns=[0,1,2,3], index=X.index)
scores = []

for train_index, test_index in grouped_cv(n_splits=2):
    X_Train, X_Test = X.iloc[train_index,:], X.iloc[test_index,:]
    y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]

    grid_estimators = []
    classes = sorted(CLASS_NAMES)   # the Status classes present in the data
    for cls in classes:
        y_Train_binary = (y_Train == cls).astype(int)

        # ── Fresh model + estimator per class, with target_class=cls ──
        # Original Model architecture with the one-vs-rest head
        dense_model = build_original_arch(N_FEATURES_RAW, n_outputs=2)

        dl_estimator = DeepLearningEstimator(dense_model, initial_weights, target_class=cls)

        grid = GridSearch(dl_estimator, constraints=EqualizedOdds(), grid_size=GS_GRID_SIZE)
        grid.fit(X_Train.values.astype('float32'),
                 y_Train_binary.values.astype('float32'),
                 sensitive_features=X_Train[['Gender','Ethnicity']])
        grid_estimators.append(grid)

    # Generate OOF predictions for each class
    for i, grid in enumerate(grid_estimators):
        y_pred_3.iloc[test_index, i] = grid.predict(X_Test.values)

    scores.append(accuracy_score(
        y_Test.values.astype('float32'),
        np.argmax(y_pred_3.loc[test_index,:].values, axis=1).astype('float32')
    ))
    print(scores[-1])

print("Mean:", np.mean(scores), "\nSTD: ", np.std(scores), '\n')

In [70]:
y_pred_3.to_csv('y_pred_gridsearch.csv',index=False)

In [71]:
results_grid = evaluate_fairness(y, y_pred_3, X, algorithm='Grid')
results_grid

=== Grid ===
Algorithm  Sensitive Feature                Status  Accuracy  Accuracy Difference  Demographic Parity Difference  Equalized Odds Difference  N Groups
     Grid             Gender Overall (all classes)     0.531                0.010                            NaN                        NaN         2
     Grid             Gender             Non-COVID     0.531                0.010                          0.030                      0.040         2
     Grid             Gender                 Covid     0.531                0.010                          0.030                      0.040         2
     Grid          Ethnicity Overall (all classes)     0.531                0.059                            NaN                        NaN         5
     Grid          Ethnicity             Non-COVID     0.531                0.059                          0.147                      0.160         5
     Grid          Ethnicity                 Covid     0.531                0.059      

In [72]:
for sf_name in SENSITIVE_SPECS:
    print(f'--- Grid Search | Sensitive Feature: {sf_name} ---')
    display(group_breakdown(y, y_pred_3, X, sensitive=sf_name))

--- Grid Search | Sensitive Feature: Gender ---
                  accuracy  selection_rate    tpr    fpr   count
Status    Gender                                                
Non-COVID Female     0.536           0.457  0.493  0.421  2880.0
          Male       0.526           0.427  0.453  0.401  3119.0
Covid     Female     0.536           0.543  0.579  0.507  2880.0
          Male       0.526           0.573  0.599  0.547  3119.0
--- Grid Search | Sensitive Feature: Ethnicity ---
                     accuracy  selection_rate    tpr    fpr   count
Status    Ethnicity                                                
Non-COVID Asian         0.506           0.467  0.475  0.459   415.0
          Black         0.565           0.475  0.554  0.429  2082.0
          Hispanic      0.505           0.522  0.544  0.512   368.0
          Others        0.541           0.547  0.589  0.507   148.0
          White         0.513           0.400  0.429  0.353  2986.0
Covid     Asian         0.506      

# Postprocessing Algorithms:

In [74]:
# Architecture: the Original Model (n_features -> 8 -> 6), with the 2-unit
# softmax one-vs-rest head this section needs.
initial_weights = original_initial_weights(N_FEATURES_RAW, n_outputs=2)


In [ ]:
from fairlearn.reductions import DemographicParity
from fairlearn.postprocessing import ThresholdOptimizer
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import clone_model
from sklearn.base import BaseEstimator

class DeepLearningEstimator(BaseEstimator):
    def __init__(self, model, initial_weights, target_class=None):
        self.model = model
        self.initial_weights = initial_weights
        self.target_class = target_class

    def __sklearn_clone__(self):
        return self.__deepcopy__({})

    def __deepcopy__(self, memo):
        new_model = clone_model(self.model)
        new_model.compile(
            optimizer=Adam(learning_rate=ORIG_LR),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        new_model.set_weights(self.initial_weights)
        return DeepLearningEstimator(new_model, self.initial_weights, self.target_class)

    def fit(self, X, y, sample_weight=None):
        y_val_binary = (y_Test == self.target_class).values.astype('float32')
        es = original_early_stopping()
        fit_es(self.model, X.astype('float32'), y.astype('float32'),
            sample_weight=sample_weight,
            epochs=ORIG_EPOCHS, batch_size=ORIG_BATCH_SIZE, verbose=0,
            callbacks=[es]
        )
        self.is_fitted_ = True

    def predict(self, X):
        # Always return a clean numpy array
        if hasattr(X, 'values'):
            X = X.values
        return (self.model(X.astype('float32')).numpy().squeeze()[:, 1] >= 0.5).astype(int)

    def predict_proba(self, X):
        # Always return a clean numpy array — never a pandas object
        if hasattr(X, 'values'):
            X = X.values
        probs = self.model(X.astype('float32')).numpy()          # pure numpy from the start
        if probs.ndim == 1:
            probs = np.stack([1 - probs, probs], axis=1)
        return probs.astype('float64')                           # float64 avoids dtype conflicts


# CV and OOF Predictions
y_pred_4 = pd.DataFrame(0.0, columns=[0,1,2,3], index=X.index)
scores = []

for train_index, test_index in grouped_cv(n_splits=5, shuffle=True, random_state=42):
    X_Train, X_Test = X.iloc[train_index,:], X.iloc[test_index,:]
    y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]

    # Always pass as numpy — eliminates ALL pandas index issues inside Fairlearn
    sf_train = X_Train[['Ethnicity', 'Gender']].values
    sf_test  = X_Test[['Ethnicity', 'Gender']].values

    threshopt_estimators = []
    classes = sorted(CLASS_NAMES)   # the Status classes present in the data
    for cls in classes:
        y_Train_binary = (y_Train == cls).astype(int)

        # ── Fresh model + estimator per class, with target_class=cls ──
        # Original Model architecture with the one-vs-rest head
        dense_model = build_original_arch(N_FEATURES_RAW, n_outputs=2)

        dl_estimator = DeepLearningEstimator(dense_model, initial_weights, target_class=cls)
        dl_estimator.fit(X_Train.values.astype('float32'),
                         y_Train_binary.values.astype('float32'))

        # Degeneracy check
        degenerate = any(
            len(set(y_Train_binary.values[np.all(sf_train == g, axis=1)])) < 2
            for g in np.unique(sf_train, axis=0)
        )

        if degenerate:
            print(f"  Class {cls}: degenerate group found, skipping ThresholdOptimizer")
            threshopt_estimators.append(dl_estimator)
        else:
            try:
                threshopt = ThresholdOptimizer(
                    estimator=dl_estimator,
                    constraints='equalized_odds',
                    prefit=True,
                    predict_method='predict_proba',
                    grid_size=5000
                )
                threshopt.fit(
                    X_Train.values.astype('float32'),
                    y_Train_binary.values.astype('float32'),
                    sensitive_features=sf_train
                )
                threshopt_estimators.append(threshopt)
            except ValueError as e:
                print(f"  Class {cls}: ThresholdOptimizer failed ({e}), using base estimator")
                threshopt_estimators.append(dl_estimator)

    # Generate OOF predictions for each class
    for i, estimator in enumerate(threshopt_estimators):
        if isinstance(estimator, ThresholdOptimizer):
            preds = estimator.predict(
                X_Test.values.astype('float32'),
                sensitive_features=sf_test,
                random_state=42
            )
        else:
            preds = estimator.predict(X_Test.values.astype('float32'))

        y_pred_4.iloc[test_index, i] = np.array(preds).astype('float32')

    scores.append(accuracy_score(
        y_Test.values.astype('float32'),
        np.argmax(y_pred_4.loc[test_index,:].values, axis=1).astype('float32')
    ))
    print(scores[-1])

print("Mean:", np.mean(scores), "\nSTD: ", np.std(scores), '\n')




# Gridsize = 1000
# 0.1845602193630036
# 0.19215355410251003
# 0.1858257751529213
# 0.18962244252267454
# 0.19194262813752372
# Mean: 0.18882092385572663 
# STD:  0.0031182819848262103 


# Gridsize = 5000
# 0.18603670111790763
# 0.1864585530478802
# 0.1883568867327568
# 0.18793503480278423
# 0.19152077620755115
# Mean: 0.188061590381776 
# STD:  0.00193592810049926 

In [76]:
y_pred_4.to_csv('y_pred_Threshold.csv',index=False)

In [77]:
results_thres = evaluate_fairness(y, y_pred_4, X, algorithm='Thres')
results_thres

=== Thres ===
Algorithm  Sensitive Feature                Status  Accuracy  Accuracy Difference  Demographic Parity Difference  Equalized Odds Difference  N Groups
    Thres             Gender Overall (all classes)     0.542                0.032                            NaN                        NaN         2
    Thres             Gender             Non-COVID     0.542                0.032                          0.022                      0.054         2
    Thres             Gender                 Covid     0.542                0.032                          0.022                      0.054         2
    Thres          Ethnicity Overall (all classes)     0.542                0.165                            NaN                        NaN         5
    Thres          Ethnicity             Non-COVID     0.542                0.165                          0.134                      0.170         5
    Thres          Ethnicity                 Covid     0.542                0.165     

In [78]:
for sf_name in SENSITIVE_SPECS:
    print(f'--- Threshold Optimizer | Sensitive Feature: {sf_name} ---')
    display(group_breakdown(y, y_pred_4, X, sensitive=sf_name))

--- Threshold Optimizer | Sensitive Feature: Gender ---
                  accuracy  selection_rate    tpr    fpr   count
Status    Gender                                                
Non-COVID Female     0.558           0.728  0.787  0.670  2880.0
          Male       0.526           0.707  0.733  0.680  3119.0
Covid     Female     0.558           0.272  0.330  0.213  2880.0
          Male       0.526           0.293  0.320  0.267  3119.0
--- Threshold Optimizer | Sensitive Feature: Ethnicity ---
                     accuracy  selection_rate    tpr    fpr   count
Status    Ethnicity                                                
Non-COVID Asian         0.542           0.798  0.819  0.773   415.0
          Black         0.488           0.711  0.775  0.675  2082.0
          Hispanic      0.429           0.717  0.737  0.709   368.0
          Others        0.520           0.838  0.863  0.813   148.0
          White         0.594           0.704  0.742  0.643  2986.0
Covid     Asian    

In [79]:
# ── Final comparison across all algorithms ──────────────────────────────────
pd.set_option('display.width', 200)

for metric in ['Accuracy', 'Accuracy Difference',
               'Demographic Parity Difference', 'Equalized Odds Difference']:
    print(f'\n===== {metric} =====')
    print(comparison_table(metric).to_string())

metrics_values.to_csv('Metrics_Values.csv', index=False)
comparison_table('Accuracy').to_csv('Comparison_Accuracy.csv')
comparison_table('Demographic Parity Difference').to_csv('Comparison_DPD.csv')
comparison_table('Equalized Odds Difference').to_csv('Comparison_EOD.csv')
metrics_values


===== Accuracy =====
Algorithm                                 Original   Corr  Expon   Grid  Thres
Sensitive Feature  Status                                                     
Gender             Overall (all classes)     0.615  0.542  0.512  0.531  0.542
                   Non-COVID                 0.615  0.542  0.512  0.531  0.542
                   Covid                     0.615  0.542  0.512  0.531  0.542
Ethnicity          Overall (all classes)     0.615  0.542  0.512  0.531  0.542
                   Non-COVID                 0.615  0.542  0.512  0.531  0.542
                   Covid                     0.615  0.542  0.512  0.531  0.542
Gender + Ethnicity Overall (all classes)     0.615  0.542  0.512  0.531  0.542
                   Non-COVID                 0.615  0.542  0.512  0.531  0.542
                   Covid                     0.615  0.542  0.512  0.531  0.542

===== Accuracy Difference =====
Algorithm                                 Original   Corr  Expon   Grid  Thr

# Sequential (Combined) Mitigation Algorithms

Two mitigation algorithms applied one after the other, in pipeline order:

1. **Correlation Remover → Exponentiated Gradient** (pre-processing → in-processing)
2. **Grid Search → Threshold Optimizer** (in-processing → post-processing)
3. **Correlation Remover → Threshold Optimizer** (pre-processing → post-processing)

Stage 1 is always fitted on the training fold only; the sensitive features passed to
stage 2 are always taken from the original (untransformed) `X`.

In [81]:
# ══════════════════════════════════════════════════════════════════════════
# SHARED HELPERS FOR SEQUENTIAL (COMBINED) MITIGATION
# ══════════════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model, clone_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

from fairlearn.preprocessing import CorrelationRemover
from fairlearn.reductions import ExponentiatedGradient, GridSearch, EqualizedOdds
from fairlearn.postprocessing import ThresholdOptimizer

# SENSITIVE_COLS, N_FEATURES_RAW and N_FEATURES_CR come from the shared
# architecture cell near the top of the Mitigation Algorithms section.
CLASSES = sorted(CLASS_NAMES)   # Status classes (y_pred_* keep their 4-column layout)


def build_binary_model(n_features, weights=None):
    """One-vs-rest binary head on the Original Model architecture
    (n_features -> 8 -> 6 -> 2 softmax). Delegates to build_original_arch so the
    sequential blocks train the same network as every other section."""
    return build_original_arch(n_features, n_outputs=2, weights=weights)


# One fixed initialisation per input width, so every fold/class starts identically
INITIAL_WEIGHTS_RAW = original_initial_weights(N_FEATURES_RAW, n_outputs=2)
INITIAL_WEIGHTS_CR  = original_initial_weights(N_FEATURES_CR,  n_outputs=2)


class DLBinaryEstimator(BaseEstimator):
    """Keras binary estimator usable by Fairlearn reductions and ThresholdOptimizer.

    Unlike the earlier DeepLearningEstimator versions this one does NOT read the
    global X_Test / y_Test for validation — it uses validation_split, so it works
    unchanged whether the input has 11 raw features or 9 correlation-removed ones.

    predict()       -> P(class = 1), continuous scores (what the reductions expect)
    predict_proba() -> [P(0), P(1)]   (what ThresholdOptimizer expects)
    """

    def __init__(self, model, initial_weights, target_class=None,
                 epochs=ORIG_EPOCHS, batch_size=ORIG_BATCH_SIZE,
                 patience=ORIG_PATIENCE, validation_split=0.2):
        self.model            = model
        self.initial_weights  = initial_weights
        self.target_class     = target_class
        self.epochs           = epochs
        self.batch_size       = batch_size
        self.patience         = patience
        self.validation_split = validation_split

    def get_params(self, deep=True):
        return {'model': self.model, 'initial_weights': self.initial_weights,
                'target_class': self.target_class, 'epochs': self.epochs,
                'batch_size': self.batch_size, 'patience': self.patience,
                'validation_split': self.validation_split}

    def set_params(self, **params):
        for k, v in params.items():
            setattr(self, k, v)
        return self

    def __sklearn_clone__(self):
        return self.__deepcopy__({})

    def __deepcopy__(self, memo):
        # ExponentiatedGradient / GridSearch deep-copy the estimator per iteration
        new_model = clone_model(self.model)
        new_model.compile(optimizer=Adam(learning_rate=ORIG_LR),
                          loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        new_model.set_weights(self.initial_weights)
        return DLBinaryEstimator(new_model, self.initial_weights, self.target_class,
                                 self.epochs, self.batch_size, self.patience,
                                 self.validation_split)

    @staticmethod
    def _as_array(X):
        return np.asarray(X.values if hasattr(X, 'values') else X).astype('float32')

    def fit(self, X, y, sample_weight=None):
        es = EarlyStopping(monitor='val_loss', mode='min', patience=self.patience,
                           restore_best_weights=True)
        fit_es(self.model, self._as_array(X),
                       np.asarray(y).astype('float32'),
                       sample_weight=None if sample_weight is None
                                     else np.asarray(sample_weight).astype('float32'),
                       epochs=self.epochs, batch_size=self.batch_size, verbose=0,
                       callbacks=[es], shuffle=True)
        self.is_fitted_ = True
        return self

    def predict(self, X):
        return self.model(self._as_array(X)).numpy()[:, 1].astype('float64')

    def predict_proba(self, X):
        probs = self.model(self._as_array(X)).numpy()
        if probs.ndim == 1:
            probs = np.stack([1 - probs, probs], axis=1)
        return probs.astype('float64')


class FittedScoreWrapper(BaseEstimator):
    """Exposes an already-fitted reduction (GridSearch / ExponentiatedGradient)
    as a prefit scorer, so ThresholdOptimizer can post-process its output."""

    def __init__(self, reduction):
        self.reduction  = reduction
        self.is_fitted_ = True

    def fit(self, X, y=None, **kwargs):
        return self          # already fitted — nothing to do

    def _scores(self, X):
        X = np.asarray(X.values if hasattr(X, 'values') else X).astype('float32')
        s = np.asarray(self.reduction.predict(X)).astype('float64').ravel()
        return np.clip(s, 0.0, 1.0)

    def predict(self, X):
        return (self._scores(X) >= 0.5).astype(int)

    def predict_proba(self, X):
        s = self._scores(X)
        return np.column_stack([1.0 - s, s])


def apply_correlation_remover(X_Train, X_Test, sensitive_cols=SENSITIVE_COLS):
    """Fit on the training fold only, transform both folds. Returns numpy arrays
    with the sensitive columns dropped (N_FEATURES_CR wide)."""
    cr = CorrelationRemover(sensitive_feature_ids=sensitive_cols)
    cr.fit(X_Train)
    return (np.asarray(cr.transform(X_Train), dtype='float32'),
            np.asarray(cr.transform(X_Test),  dtype='float32'),
            cr)


def has_degenerate_group(y_binary, sf_array):
    """True if any sensitive group contains only one label — ThresholdOptimizer
    cannot solve the ROC problem in that case."""
    y_binary = np.asarray(y_binary).ravel()
    sf_array = np.asarray(sf_array)
    for g in np.unique(sf_array, axis=0):
        mask = np.all(sf_array == g, axis=1)
        if len(np.unique(y_binary[mask])) < 2:
            return True
    return False


def fold_accuracy(y_true_fold, pred_block):
    return accuracy_score(np.asarray(y_true_fold).astype(int),
                          np.argmax(np.asarray(pred_block), axis=1))

## Correlation Remover + Exponentiated Gradient

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 1) CORRELATION REMOVER  ->  EXPONENTIATED GRADIENT
#    pre-processing decorrelates the features, the reduction then enforces
#    Equalized Odds during training on those decorrelated features.
# ══════════════════════════════════════════════════════════════════════════
y_pred_5 = pd.DataFrame(0.0, columns=[0, 1, 2, 3], index=X.index)
scores = []

for train_index, test_index in grouped_cv(n_splits=2):
    X_Train, X_Test = X.iloc[train_index, :], X.iloc[test_index, :]
    y_Train, y_Test = y.iloc[train_index],    y.iloc[test_index]

    # ── Stage 1: Correlation Remover (fit on train fold only) ──
    X_Train_cr, X_Test_cr, _ = apply_correlation_remover(X_Train, X_Test)

    # sensitive features come from the ORIGINAL frame — they no longer exist in *_cr
    sf_train = X_Train[SENSITIVE_COLS]
    sf_test  = X_Test[SENSITIVE_COLS]

    # ── Stage 2: Exponentiated Gradient, one-vs-rest ──
    for i, cls in enumerate(CLASSES):
        y_Train_binary = (y_Train == cls).astype(int).values.astype('float32')

        model = build_binary_model(N_FEATURES_CR, INITIAL_WEIGHTS_CR)
        dl_estimator = DLBinaryEstimator(model, INITIAL_WEIGHTS_CR,
                                         target_class=cls)   # original schedule

        expgrad = ExponentiatedGradient(dl_estimator,
                                        constraints=EqualizedOdds(), max_iter=10)
        expgrad.fit(X_Train_cr, y_Train_binary, sensitive_features=sf_train)

        y_pred_5.iloc[test_index, i] = np.asarray(expgrad.predict(X_Test_cr)).astype('float32')

    scores.append(fold_accuracy(y_Test.values, y_pred_5.iloc[test_index, :].values))
    print(scores[-1])

print("Mean:", np.mean(scores), "\nSTD: ", np.std(scores), '\n')

In [84]:
y_pred_5.to_csv('y_pred_Corr_ExpGrad.csv', index=False)

In [85]:
results_corr_expon = evaluate_fairness(y, y_pred_5, X, algorithm='Corr+Expon')
results_corr_expon

=== Corr+Expon ===
 Algorithm  Sensitive Feature                Status  Accuracy  Accuracy Difference  Demographic Parity Difference  Equalized Odds Difference  N Groups
Corr+Expon             Gender Overall (all classes)     0.515                0.020                            NaN                        NaN         2
Corr+Expon             Gender             Non-COVID     0.515                0.020                          0.002                      0.022         2
Corr+Expon             Gender                 Covid     0.515                0.020                          0.002                      0.022         2
Corr+Expon          Ethnicity Overall (all classes)     0.515                0.123                            NaN                        NaN         5
Corr+Expon          Ethnicity             Non-COVID     0.515                0.123                          0.031                      0.094         5
Corr+Expon          Ethnicity                 Covid     0.515              

In [86]:
for sf_name in SENSITIVE_SPECS:
    print(f'--- Correlation Remover + Exponentiated Gradient | Sensitive Feature: {sf_name} ---')
    display(group_breakdown(y, y_pred_5, X, sensitive=sf_name))

--- Correlation Remover + Exponentiated Gradient | Sensitive Feature: Gender ---
                  accuracy  selection_rate    tpr    fpr   count
Status    Gender                                                
Non-COVID Female     0.526           0.761  0.787  0.735  2880.0
          Male       0.506           0.759  0.764  0.753  3119.0
Covid     Female     0.526           0.239  0.265  0.213  2880.0
          Male       0.506           0.241  0.247  0.236  3119.0
--- Correlation Remover + Exponentiated Gradient | Sensitive Feature: Ethnicity ---
                     accuracy  selection_rate    tpr    fpr   count
Status    Ethnicity                                                
Non-COVID Asian         0.533           0.754  0.769  0.737   415.0
          Black         0.446           0.746  0.765  0.735  2082.0
          Hispanic      0.446           0.777  0.860  0.740   368.0
          Others        0.547           0.770  0.822  0.720   148.0
          White         0.568        

## Grid Search + Threshold Optimizer

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 2) GRID SEARCH  ->  THRESHOLD OPTIMIZER
#    the reduction produces a constrained predictor, which is then wrapped as a
#    prefit scorer and post-processed with group-specific thresholds.
# ══════════════════════════════════════════════════════════════════════════
y_pred_6 = pd.DataFrame(0.0, columns=[0, 1, 2, 3], index=X.index)
scores = []

for train_index, test_index in grouped_cv(n_splits=2):
    X_Train, X_Test = X.iloc[train_index, :], X.iloc[test_index, :]
    y_Train, y_Test = y.iloc[train_index],    y.iloc[test_index]

    X_Train_np = X_Train.values.astype('float32')
    X_Test_np  = X_Test.values.astype('float32')
    sf_train   = X_Train[SENSITIVE_COLS].values      # numpy avoids index issues
    sf_test    = X_Test[SENSITIVE_COLS].values

    for i, cls in enumerate(CLASSES):
        y_Train_binary = (y_Train == cls).astype(int).values.astype('float32')

        # ── Stage 1: Grid Search ──
        model = build_binary_model(N_FEATURES_RAW, INITIAL_WEIGHTS_RAW)
        dl_estimator = DLBinaryEstimator(model, INITIAL_WEIGHTS_RAW,
                                         target_class=cls)   # original schedule

        grid = GridSearch(dl_estimator, constraints=EqualizedOdds(), grid_size=GS_GRID_SIZE)
        grid.fit(X_Train_np, y_Train_binary, sensitive_features=sf_train)

        # ── Stage 2: Threshold Optimizer on top of the fitted Grid Search ──
        scorer = FittedScoreWrapper(grid)

        if has_degenerate_group(y_Train_binary, sf_train):
            print(f"  Class {cls}: degenerate group found, skipping ThresholdOptimizer")
            preds = scorer.predict_proba(X_Test_np)[:, 1]
        else:
            try:
                threshopt = ThresholdOptimizer(estimator=scorer,
                                               constraints='equalized_odds',
                                               prefit=True,
                                               predict_method='predict_proba',
                                               grid_size=5000)
                threshopt.fit(X_Train_np, y_Train_binary, sensitive_features=sf_train)
                preds = threshopt.predict(X_Test_np, sensitive_features=sf_test,
                                          random_state=42)
            except ValueError as e:
                print(f"  Class {cls}: ThresholdOptimizer failed ({e}), using GridSearch output")
                preds = scorer.predict_proba(X_Test_np)[:, 1]

        y_pred_6.iloc[test_index, i] = np.asarray(preds).astype('float32')

    scores.append(fold_accuracy(y_Test.values, y_pred_6.iloc[test_index, :].values))
    print(scores[-1])

print("Mean:", np.mean(scores), "\nSTD: ", np.std(scores), '\n')

In [89]:
y_pred_6.to_csv('y_pred_Grid_Threshold.csv', index=False)

In [90]:
results_grid_thres = evaluate_fairness(y, y_pred_6, X, algorithm='Grid+Thres')
results_grid_thres

=== Grid+Thres ===
 Algorithm  Sensitive Feature                Status  Accuracy  Accuracy Difference  Demographic Parity Difference  Equalized Odds Difference  N Groups
Grid+Thres             Gender Overall (all classes)     0.507                0.001                            NaN                        NaN         2
Grid+Thres             Gender             Non-COVID     0.507                0.001                          0.002                      0.004         2
Grid+Thres             Gender                 Covid     0.507                0.001                          0.002                      0.004         2
Grid+Thres          Ethnicity Overall (all classes)     0.507                0.196                            NaN                        NaN         5
Grid+Thres          Ethnicity             Non-COVID     0.507                0.196                          0.065                      0.067         5
Grid+Thres          Ethnicity                 Covid     0.507              

In [91]:
for sf_name in SENSITIVE_SPECS:
    print(f'--- Grid Search + Threshold Optimizer | Sensitive Feature: {sf_name} ---')
    display(group_breakdown(y, y_pred_6, X, sensitive=sf_name))

--- Grid Search + Threshold Optimizer | Sensitive Feature: Gender ---
                  accuracy  selection_rate    tpr    fpr   count
Status    Gender                                                
Non-COVID Female     0.507           0.801  0.808  0.793  2880.0
          Male       0.506           0.803  0.809  0.797  3119.0
Covid     Female     0.507           0.199  0.207  0.192  2880.0
          Male       0.506           0.197  0.203  0.191  3119.0
--- Grid Search + Threshold Optimizer | Sensitive Feature: Ethnicity ---
                     accuracy  selection_rate    tpr    fpr   count
Status    Ethnicity                                                
Non-COVID Asian         0.525           0.843  0.846  0.840   415.0
          Black         0.430           0.779  0.788  0.773  2082.0
          Hispanic      0.378           0.840  0.851  0.835   368.0
          Others        0.507           0.824  0.836  0.813   148.0
          White         0.574           0.807  0.809  0.803

## Correlation Remover + Threshold Optimizer

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 3) CORRELATION REMOVER  ->  THRESHOLD OPTIMIZER
#    decorrelated features feed an unconstrained model, whose scores are then
#    thresholded per sensitive group.
# ══════════════════════════════════════════════════════════════════════════
y_pred_7 = pd.DataFrame(0.0, columns=[0, 1, 2, 3], index=X.index)
scores = []

for train_index, test_index in grouped_cv(n_splits=5, shuffle=True,
                                               random_state=42):
    X_Train, X_Test = X.iloc[train_index, :], X.iloc[test_index, :]
    y_Train, y_Test = y.iloc[train_index],    y.iloc[test_index]

    # ── Stage 1: Correlation Remover ──
    X_Train_cr, X_Test_cr, _ = apply_correlation_remover(X_Train, X_Test)
    sf_train = X_Train[SENSITIVE_COLS].values
    sf_test  = X_Test[SENSITIVE_COLS].values

    for i, cls in enumerate(CLASSES):
        y_Train_binary = (y_Train == cls).astype(int).values.astype('float32')

        # base model trained on the decorrelated features
        model = build_binary_model(N_FEATURES_CR, INITIAL_WEIGHTS_CR)
        dl_estimator = DLBinaryEstimator(model, INITIAL_WEIGHTS_CR,
                                         target_class=cls)   # original schedule
        dl_estimator.fit(X_Train_cr, y_Train_binary)

        # ── Stage 2: Threshold Optimizer (prefit) ──
        if has_degenerate_group(y_Train_binary, sf_train):
            print(f"  Class {cls}: degenerate group found, skipping ThresholdOptimizer")
            preds = dl_estimator.predict(X_Test_cr)
        else:
            try:
                threshopt = ThresholdOptimizer(estimator=dl_estimator,
                                               constraints='equalized_odds',
                                               prefit=True,
                                               predict_method='predict_proba',
                                               grid_size=5000)
                threshopt.fit(X_Train_cr, y_Train_binary, sensitive_features=sf_train)
                preds = threshopt.predict(X_Test_cr, sensitive_features=sf_test,
                                          random_state=42)
            except ValueError as e:
                print(f"  Class {cls}: ThresholdOptimizer failed ({e}), using base estimator")
                preds = dl_estimator.predict(X_Test_cr)

        y_pred_7.iloc[test_index, i] = np.asarray(preds).astype('float32')

    scores.append(fold_accuracy(y_Test.values, y_pred_7.iloc[test_index, :].values))
    print(scores[-1])

print("Mean:", np.mean(scores), "\nSTD: ", np.std(scores), '\n')

In [94]:
y_pred_7.to_csv('y_pred_Corr_Threshold.csv', index=False)

In [95]:
results_corr_thres = evaluate_fairness(y, y_pred_7, X, algorithm='Corr+Thres')
results_corr_thres

=== Corr+Thres ===
 Algorithm  Sensitive Feature                Status  Accuracy  Accuracy Difference  Demographic Parity Difference  Equalized Odds Difference  N Groups
Corr+Thres             Gender Overall (all classes)     0.529                0.001                            NaN                        NaN         2
Corr+Thres             Gender             Non-COVID     0.529                0.001                          0.014                      0.015         2
Corr+Thres             Gender                 Covid     0.529                0.001                          0.014                      0.015         2
Corr+Thres          Ethnicity Overall (all classes)     0.529                0.098                            NaN                        NaN         5
Corr+Thres          Ethnicity             Non-COVID     0.529                0.098                          0.114                      0.133         5
Corr+Thres          Ethnicity                 Covid     0.529              

In [96]:
for sf_name in SENSITIVE_SPECS:
    print(f'--- Correlation Remover + Threshold Optimizer | Sensitive Feature: {sf_name} ---')
    display(group_breakdown(y, y_pred_7, X, sensitive=sf_name))

--- Correlation Remover + Threshold Optimizer | Sensitive Feature: Gender ---
                  accuracy  selection_rate    tpr    fpr   count
Status    Gender                                                
Non-COVID Female     0.530           0.643  0.673  0.613  2880.0
          Male       0.529           0.629  0.658  0.600  3119.0
Covid     Female     0.530           0.357  0.387  0.327  2880.0
          Male       0.529           0.371  0.400  0.342  3119.0
--- Correlation Remover + Threshold Optimizer | Sensitive Feature: Ethnicity ---
                     accuracy  selection_rate    tpr    fpr   count
Status    Ethnicity                                                
Non-COVID Asian         0.506           0.704  0.697  0.711   415.0
          Black         0.498           0.638  0.687  0.610  2082.0
          Hispanic      0.481           0.590  0.614  0.579   368.0
          Others        0.466           0.676  0.644  0.707   148.0
          White         0.564           0.6

## Final Comparison

In [98]:
# ── Final comparison: single vs. sequential mitigation ──────────────────────
pd.set_option('display.width', 250)

ALGO_ORDER = ['Original', 'Corr', 'Expon', 'Grid', 'Thres',
              'Corr+Expon', 'Grid+Thres', 'Corr+Thres']

def ordered_comparison(metric, decimals=3):
    tbl = comparison_table(metric, decimals)
    cols = [a for a in ALGO_ORDER if a in tbl.columns]
    cols += [c for c in tbl.columns if c not in cols]
    return tbl[cols]

for metric in ['Accuracy', 'Accuracy Difference',
               'Demographic Parity Difference', 'Equalized Odds Difference']:
    print(f'\n===== {metric} =====')
    print(ordered_comparison(metric).to_string())

metrics_values.to_csv('Metrics_Values_Fairlear.csv', index=False)
ordered_comparison('Accuracy').to_csv('Comparison_Accuracy_Fairlear.csv')
ordered_comparison('Demographic Parity Difference').to_csv('Comparison_DPD_Fairlear.csv')
ordered_comparison('Equalized Odds Difference').to_csv('Comparison_EOD_Fairlear.csv')
metrics_values


===== Accuracy =====
Algorithm                                 Original   Corr  Expon   Grid  Thres  Corr+Expon  Grid+Thres  Corr+Thres
Sensitive Feature  Status                                                                                         
Gender             Overall (all classes)     0.615  0.542  0.512  0.531  0.542       0.515       0.507       0.529
                   Non-COVID                 0.615  0.542  0.512  0.531  0.542       0.515       0.507       0.529
                   Covid                     0.615  0.542  0.512  0.531  0.542       0.515       0.507       0.529
Ethnicity          Overall (all classes)     0.615  0.542  0.512  0.531  0.542       0.515       0.507       0.529
                   Non-COVID                 0.615  0.542  0.512  0.531  0.542       0.515       0.507       0.529
                   Covid                     0.615  0.542  0.512  0.531  0.542       0.515       0.507       0.529
Gender + Ethnicity Overall (all classes)     0.615  0.542 